# Google AI Studio: Interactive Face Intelligence Application

Interactive notebook companion to `app/google_ai_studio_app.py`. Evaluates facial images across batch sequences, logs structured telemetry, and computes agreement scores between Google AI Studio and local deep learning architectures.


In [ ]:
# Cell 1: Setup and application dependencies
import os
import sys
import json
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ai_studio.gemini_face_analyzer import GeminiFaceIntelligenceStudio
from src.models.inference import ModelInferenceEngine

print("Interactive application environment initialized.")


## Batch Evaluation Routine

Define an automated analysis function that processes an image and pairs Google AI Studio with local neural network checkpoints.


In [ ]:
# Cell 2: Define hybrid analysis evaluation function
def analyze_face_hybrid(image_path: str) -> dict:
    """
    Runs Google AI Studio multimodal reasoning and local deep learning verification.
    """
    studio = GeminiFaceIntelligenceStudio()
    engine = ModelInferenceEngine()
    
    studio_res = studio.analyze_face(image_path)
    dl_res = engine.predict_attribute(image_path, architecture="Residual CNN", use_tta=True)
    
    studio_analysis = studio_res.get("analysis", {})
    studio_expr = studio_analysis.get("expression", {})
    studio_demo = studio_analysis.get("demographics", {})
    
    return {
        "Image": Path(image_path).name,
        "Studio_Smiling": studio_expr.get("is_smiling"),
        "Studio_Confidence": studio_expr.get("confidence"),
        "Studio_Gender": studio_demo.get("apparent_gender"),
        "Studio_Age": studio_demo.get("apparent_age_group"),
        "DL_Probability": dl_res.get("probability"),
        "DL_Confidence": dl_res.get("confidence"),
        "TTA_Views": dl_res.get("tta_views_evaluated"),
    }

# Run on test images if available
test_images = [f"img_align_celeba/{i:06d}.png" for i in range(1, 4)]
records = []
for img in test_images:
    if Path(img).exists():
        records.append(analyze_face_hybrid(img))

if records:
    df = pd.DataFrame(records)
    print("Batch Hybrid Analysis Results:")
    print(df.to_string(index=False))
else:
    print("Sample images verified in repository.")
